# Lab 04-04 — Parent-child retrieval: small chunks in, large context out

**Track 04 · Retrieval** — plain chunked retrieval has a tension: small chunks match a query precisely but carry too little surrounding context for an answer; large chunks carry context but match a query coarsely. Parent-child retrieval splits the difference — embed SMALL child chunks for precise matching, but store the LARGER parent document each child came from, and return the PARENT whenever a child hits.

This notebook is **self-contained**: it imports LangChain, sentence-transformers, and faiss directly — no repo component library. Every block of the pipeline is built right here: the two `RecursiveCharacterTextSplitter`s (small children, large parents), the local BGE embedder, the FAISS store over the children, the `InMemoryStore` docstore over the parents, and the `ParentDocumentRetriever` that links the two — which is exactly how the shared components in `src/` work underneath.

This lab builds a `ParentDocumentRetriever` (from `langchain-classic`, the retriever home of the LangChain 1.x era) over a deterministic subset of `Data/corpus/rag-mini-wikipedia`:

* **PARENTS** — first 20 passages of the corpus, re-split at `PARENT_CHUNK_SIZE` (500 chars): these are the large contexts that get returned.
* **CHILDREN** — each parent split again at `CHILD_CHUNK_SIZE` (120 chars): these are the small pieces that get embedded and matched.
* The retriever keeps a `docstore` mapping every child's `doc_id` back to its parent, so a query that matches a 120-char child returns the whole ~500-char parent.

Local embeddings only (BGE via sentence-transformers); no LLM, no API keys. The vector store is FAISS in-memory — nothing is written to disk.


## Setup

Two prerequisites must hold before this notebook will run:

- **rag-mini-wikipedia on disk** — `Data/corpus/rag-mini-wikipedia/` (`passages.parquet` + `test.parquet`), already fetched by the repo's manifest-verified fetchers.

No repo imports are needed: everything this notebook uses comes from `langchain-core`, `langchain-classic`, `langchain-community`, `langchain-huggingface`, `langchain-text-splitters`, `sentence-transformers`, `faiss-cpu`, and `pandas`. The imports cell walks up to the repo root and cd's into it, because a notebook has no `__file__` — so every `Data/...` path resolves exactly like the lab script. Unlike the Curriculum notebook, there is no `sys.path` trick: nothing is imported from `src/`.

The next cell installs the notebook-specific dependencies (a no-op if you already ran `pip install -r requirements.txt`).


In [ ]:
# Lab-specific dependencies (already in requirements.txt — the install
# below is a no-op if you have run `pip install -r requirements.txt`).
#   sentence-transformers -> local BGE embeddings (langchain_huggingface)
#   faiss-cpu             -> the FAISS vector store (langchain_community)
#   langchain-classic     -> ParentDocumentRetriever
#   langchain-text-splitters -> the child/parent RecursiveCharacterTextSplitter
#   pandas                -> reads the passages/test.parquet corpus
%pip install -q sentence-transformers langchain-huggingface langchain-community langchain-classic langchain-text-splitters faiss-cpu pandas


In [ ]:
# Bootstrap: stdlib imports + repo-root walk (no sys.path tricks).
from __future__ import annotations

import os
import time
from pathlib import Path

import pandas as pd

# LangChain + sentence-transformers + faiss — the only libraries this
# notebook needs. Nothing is imported from the repo's src/ component library.
from langchain_classic.retrievers import ParentDocumentRetriever  # noqa: E402
from langchain_community.vectorstores import FAISS  # noqa: E402
from langchain_core.documents import Document  # noqa: E402
from langchain_core.stores import InMemoryStore  # noqa: E402
from langchain_huggingface import HuggingFaceEmbeddings  # noqa: E402
from langchain_text_splitters import RecursiveCharacterTextSplitter  # noqa: E402

# A notebook has no __file__, so walk up from the cwd to the repo root and
# cd into it — Data/... paths then resolve exactly like the lab script.
REPO_ROOT = Path.cwd()
for _candidate in [Path.cwd(), *Path.cwd().parents]:
    if (_candidate / "src" / "curriculum").is_dir() and (_candidate / "NoteBooks").is_dir():
        REPO_ROOT = _candidate
        break
os.chdir(REPO_ROOT)


## 1. Configuration

Everything that keeps this lab fast but still meaningful is a named constant. `N_PARENTS = 20` takes the deterministic head of the 3200-passage corpus; `QUESTION_IDS = [1606, 1610, 1604]` are real questions whose answers live inside the subset. The splitter geometry is the whole lab: `CHILD_CHUNK_SIZE = 120` (the small chunks that get embedded and matched) with `CHILD_OVERLAP = 20`, `PARENT_CHUNK_SIZE = 500` (the large contexts returned to the caller) with `PARENT_OVERLAP = 50`, and `K = 3` — how many parents each query asks for.


In [ ]:
# --------------------------------------------------------------------------
# 1. Configuration — tweak these to rerun the experiment
# --------------------------------------------------------------------------
PASSAGES_PATH = Path("Data/corpus/rag-mini-wikipedia/passages.parquet")
TEST_PATH = Path("Data/corpus/rag-mini-wikipedia/test.parquet")
N_PARENTS = 20  # deterministic head of the 3200-passage corpus (keeps runtime low)
QUESTION_IDS = [1606, 1610, 1604]  # real questions from test.parquet, answers inside the subset
CHILD_CHUNK_SIZE = 120  # small chunks: embedded and matched against the query
CHILD_OVERLAP = 20
PARENT_CHUNK_SIZE = 500  # large contexts: returned to the caller
PARENT_OVERLAP = 50
K = 3  # search_kwargs: how many parents each query returns
BGE_MODEL_NAME = "BAAI/bge-base-en-v1.5"
PREVIEW = 62  # max characters of chunk text shown next to each hit


## 2. Load — parents + questions from the fresh parquet files

`load_parents` returns the first `n` passages as parent Documents with a string `doc_id` in metadata — the key the docstore links children to; `load_questions` pulls specific test rows by id; `preview` flattens a chunk for one-line printing.


In [ ]:
# --------------------------------------------------------------------------
# 2. Load — corpus + questions from the fresh rag-mini-wikipedia parquet files
# --------------------------------------------------------------------------
def load_parents(path: Path, n: int) -> list[Document]:
    """Return the first ``n`` passages as parent Documents (id -> doc_id)."""
    df = pd.read_parquet(path)
    subset = df.head(n)
    return [
        Document(page_content=text, metadata={"doc_id": str(i)})
        for i, text in enumerate(subset["passage"].tolist())
    ]


def load_questions(path: Path, ids: list[int]) -> list[tuple[int, str]]:
    """Return [(question_id, question_text)] for the requested test rows."""
    df = pd.read_parquet(path)
    rows = df.loc[ids]
    return [(int(idx), row["question"]) for idx, row in rows.iterrows()]


def preview(text: str, limit: int = PREVIEW) -> str:
    """Flatten a chunk for one-line printing."""
    flat = text.replace("\n", " ")
    return flat[:limit] + ("..." if len(flat) > limit else "")


## 3. Experiment — split into children, embed, index, query

The whole pipeline is built inline. Two `RecursiveCharacterTextSplitter`s — one for children (120 chars), one for parents (500 chars) — plus local BGE embeddings. FAISS cannot infer the embedding dimension from an empty list, so the store is seeded with one real parent and the seed deleted again (the index stays empty); the `InMemoryStore` docstore holds the parents keyed by `doc_id`. `ParentDocumentRetriever` wires the two together: `add_documents` splits each parent into children, embeds the children into FAISS, and stores the parents in the docstore. Every query then `invoke()`s the retriever to get K parents, while a direct `similarity_search_with_score` on the store shows the CHILD that actually matched. `run_experiment` returns a dict of artifacts instead of printing, so the demo and the gate read the same run.


In [ ]:
# --------------------------------------------------------------------------
# 3. Experiment — split into children, embed, index, query; returns every
#    artifact the demo and the verification gate need (no re-computation
#    between the two paths)
# --------------------------------------------------------------------------
def run_experiment() -> dict:
    parents = load_parents(PASSAGES_PATH, N_PARENTS)
    questions = load_questions(TEST_PATH, QUESTION_IDS)

    # --- Splitters: one for parents (big), one for children (small) ---------
    child_splitter = RecursiveCharacterTextSplitter(
        chunk_size=CHILD_CHUNK_SIZE, chunk_overlap=CHILD_OVERLAP
    )
    parent_splitter = RecursiveCharacterTextSplitter(
        chunk_size=PARENT_CHUNK_SIZE, chunk_overlap=PARENT_OVERLAP
    )

    # --- Local BGE embeddings (never an API model) --------------------------
    embeddings = HuggingFaceEmbeddings(
        model_name=BGE_MODEL_NAME,
        encode_kwargs={"normalize_embeddings": True},
    )

    # --- In-memory stores ---------------------------------------------------
    # FAISS cannot infer the embedding dimension from an empty list, so seed
    # it with one real parent and delete the seed again (index stays empty).
    seed = FAISS.from_documents([parents[0]], embedding=embeddings)
    seed_id = next(iter(seed.index_to_docstore_id.values()))
    seed.delete([seed_id])
    vs = seed  # holds the CHILDREN
    docstore = InMemoryStore()  # holds the PARENTS, keyed by doc_id

    retriever = ParentDocumentRetriever(
        vectorstore=vs,
        docstore=docstore,
        child_splitter=child_splitter,
        parent_splitter=parent_splitter,
        search_kwargs={"k": K},
    )

    # --- Split + embed + index in one call ----------------------------------
    t0 = time.perf_counter()
    retriever.add_documents(parents, add_to_docstore=True)
    ingest_s = time.perf_counter() - t0

    # Children live inside the FAISS vector store, keyed by index position in
    # index_to_docstore_id; every child knows its parent via metadata.doc_id.
    child_ids = list(vs.index_to_docstore_id.values())
    children = vs.get_by_ids(child_ids)
    parent_ids = list(docstore.yield_keys())
    id_to_parent = dict(zip(parent_ids, docstore.mget(parent_ids)))

    # --- Query: invoke() returns PARENTS; a direct vector search shows the
    #     CHILD that matched ------------------------------------------------
    results = []
    for qid, qtext in questions:
        retrieved = retriever.invoke(qtext)  # K parents
        matched_child, score = vs.similarity_search_with_score(qtext, k=1)[0]
        results.append(
            {
                "qid": qid,
                "qtext": qtext,
                "parents": retrieved,
                "child": matched_child,
                "child_score": score,
                "child_parent": id_to_parent.get(matched_child.metadata.get("doc_id")),
            }
        )

    parent_lens = [len(d.page_content) for d in id_to_parent.values()]
    child_lens = [len(c.page_content) for c in children]

    return {
        "n_parents": len(parent_ids),
        "n_children": len(children),
        "children": children,
        "parent_lens": parent_lens,
        "child_lens": child_lens,
        "results": results,
        "ingest_s": ingest_s,
        "n_queries": len(questions),
    }


## 4. Demo — print the artifact

`print_demo(exp)` prints the artifact from three angles: the corpus stats — parents -> children counts, mean parent/child lengths and their ratio; per question, the top-K PARENTS with their character counts, the matched CHILD with its score, and the `child -> parent` context-multiplier line showing how much larger the returned context is than the chunk that matched; then a takeaway framing the "small chunks in, large context out" trick.


In [ ]:
# --------------------------------------------------------------------------
# 4. Demo — print the artifact
# --------------------------------------------------------------------------
def print_demo(exp: dict) -> None:
    n_parents, n_children = exp["n_parents"], exp["n_children"]
    mean_p = sum(exp["parent_lens"]) / len(exp["parent_lens"])
    mean_c = sum(exp["child_lens"]) / len(exp["child_lens"])

    print("=" * 66)
    print("Lab 04 — Parent-child retrieval: small chunks in, large context out")
    print(f"{BGE_MODEL_NAME} | FAISS (in-memory) | ParentDocumentRetriever")
    print("=" * 66)

    print(f"\n[1] Corpus (deterministic subset, no randomness):")
    print(f"    {n_parents} parents (first {N_PARENTS} passages of 3200, split at {PARENT_CHUNK_SIZE} chars)")
    print(f"    {n_children} children (embedded at {CHILD_CHUNK_SIZE} chars)")
    print(f"    {exp['n_queries']} questions from test.parquet:")
    for r in exp["results"]:
        print(f"      [{r['qid']}] {r['qtext']}")

    print(f"\n[2] Split + embed + index:")
    print(f"    {n_parents} parents -> {n_children} children in {exp['ingest_s']:.2f}s")
    print(f"    mean parent length {mean_p:.0f} chars vs mean child length {mean_c:.0f} chars"
          f" ({mean_p / mean_c:.1f}x)")
    print(f"    children >> parents: {n_children} child vectors indexed, "
          f"{n_parents} parent docs in the docstore")

    print(f"\n[3] Top-{K} per question (each hit is a PARENT, matched via its children):")
    for r in exp["results"]:
        print(f'\n    Q[{r["qid"]}] "{r["qtext"]}"')
        for rank, doc in enumerate(r["parents"], 1):
            print(f"      {rank}. PARENT ({len(doc.page_content)} chars) "
                  f"[{preview(doc.page_content)}]")
        child = r["child"]
        print(f"      matched CHILD ({len(child.page_content)} chars, "
              f"score {r['child_score']:.4f}):")
        print(f"        [{preview(child.page_content)}]")
        cp = r["child_parent"]
        if cp is not None:
            print(f"      child -> parent ({len(cp.page_content)} chars): the returned "
                  f"context is {len(cp.page_content) / max(len(child.page_content), 1):.1f}x "
                  f"the chunk that matched")

    print("\n[4] Takeaway")
    print("    The retriever embeds 120-char children for precise matching, then")
    print("    maps each hit back through metadata.doc_id to its ~500-char parent.")
    print("    Precision comes from the small chunk, context from the large one —")
    print("    the 'small chunks in, large context out' trick of Project 09.")


## 5. Verification gate

`verify_gate(exp)` enforces the lab's hard checks: children outnumber parents; no empty child chunks; every retrieved hit is a PARENT (longer than `CHILD_CHUNK_SIZE`); each question returns between 1 and `K` parents (ParentDocumentRetriever dedupes — several top children can belong to the same parent, so the count is at most K but not always exactly K); the content checks (Q1610's top-1 parent names the Spanish founder of Montevideo, Q1606's top-1 parent mentions Montevideo); and the linkage check — the matched child text is literally contained in the docstore parent it claims to come from. This is the same gate the CI-style `--verify` run applies; every check should print PASS.


In [ ]:
# --------------------------------------------------------------------------
# 5. Verification gate
# --------------------------------------------------------------------------
def verify_gate(exp: dict) -> int:
    checks: list[tuple[str, bool]] = []

    # The child splitter actually split: many small chunks from few parents.
    checks.append((f"children ({exp['n_children']}) outnumber parents ({exp['n_parents']})",
                   exp["n_children"] > exp["n_parents"]))
    checks.append(("no empty child chunks",
                   all(len(c.page_content) > 0 for c in exp["children"])))

    # Every retrieved hit is a PARENT (longer than any child chunk)...
    all_parents = all(
        len(d.page_content) > CHILD_CHUNK_SIZE
        for r in exp["results"] for d in r["parents"]
    )
    checks.append(("every retrieved doc is a parent (len > CHILD_CHUNK_SIZE)",
                   all_parents))

    # ...and each question returns between 1 and K parents. ParentDocumentRetriever
    # dedupes: several top children can belong to the same parent, so the count
    # is at most K but not always exactly K.
    checks.append((f"each question returns 1..{K} parents (deduped)",
                   all(1 <= len(r["parents"]) <= K for r in exp["results"])))

    # Content: Q1610 "Who founded Montevideo?" must retrieve the parent that
    # says the Spanish founded Montevideo.
    q1610_top = exp["results"][1]["parents"][0].page_content.lower()
    checks.append(("Q1610 top-1 parent names the Spanish founder of Montevideo",
                   "spanish" in q1610_top))

    # Q1606 "Is Uruguay's capital Montevideo?" must retrieve a Uruguay parent
    # that mentions Montevideo.
    q1606_top = exp["results"][0]["parents"][0].page_content.lower()
    checks.append(("Q1606 top-1 parent mentions Montevideo", "montevideo" in q1606_top))

    # Linkage: the matched child is literally a fragment of the parent the
    # docstore says it came from (deterministic — the splitter concatenates
    # substrings, it never rewrites text).
    linked = all(
        r["child_parent"] is not None
        and r["child"].page_content.strip() in r["child_parent"].page_content
        for r in exp["results"]
    )
    checks.append(("matched child text is contained in its docstore parent",
                   linked))

    print("verification gate:")
    for label, ok in checks:
        print(f"  [{'PASS' if ok else 'FAIL'}] {label}")
    return 0 if all(ok for _, ok in checks) else 1


## Run the experiment

A couple of minutes of splitting + embedding (~80 children) + indexing — no downloads, no API calls. `exp` holds everything the demo and gate need.


In [ ]:
exp = run_experiment()


### Demo — the artifact

The parents/children counts with length ratio, and per question the top-K PARENTS with the matched CHILD and its parent-linkage multiplier.


In [ ]:
print_demo(exp)


### Verification gate

Expect every check to PASS — the same gate the CI-style `--verify` run enforces. If any line shows FAIL, check the parquet files are intact.


In [ ]:
verify_gate(exp)
